# Simpson's paradox và funnel: khi con số gộp kể sai câu chuyện

Hai chart hay gặp nhất trong báo cáo cho lãnh đạo là **tỉ lệ gộp** ("tỉ lệ X của nhóm A thấp hơn nhóm B")
và **funnel** ("bao nhiêu % đi tới bước cuối"). Cả hai đều là phép chia, và cả hai đều dễ đọc sai:

- **Phần A, Simpson's paradox** trên số liệu tuyển sinh sau đại học UC Berkeley mùa thu 1973, ví dụ kinh
  điển trong thống kê: tỉ lệ nhận hồ sơ gộp của nữ thấp hơn nam rõ rệt, nhưng xét từng khoa thì bức tranh
  khác hẳn. Bạn tính cả hai, rồi viết hàm phát hiện đảo chiều.
- **Phần B, funnel đơn hàng** trên bộ Olist thật (99 441 đơn, 2016-2018): mua → duyệt thanh toán → giao
  cho hãng vận chuyển → giao tới khách. Bạn dựng funnel, tính tỉ lệ chuyển đổi từng bước, và đọc nó như
  một PM sẽ đọc.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/intermediate/03-simpson-and-funnel -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Phần A. Tuyển sinh UC Berkeley 1973

Nguồn: P. J. Bickel, E. A. Hammel, J. W. O'Connell, "Sex Bias in Graduate Admissions: Data from
Berkeley", *Science* 187 (4175): 398-404, 1975. Bảng dưới là 6 khoa lớn nhất (bài báo ẩn tên khoa, gọi
A-F), cùng số liệu với bộ `UCBAdmissions` có sẵn trong R. Mỗi dòng: khoa, giới, số hồ sơ nộp, số hồ sơ
được nhận. Hằng `BERKELEY` đã nhập sẵn, đừng sửa: test kiểm lại các tổng (nam 1 198/2 691, nữ 557/1 835).

`admit_rate(df, by) -> pd.Series`: tỉ lệ nhận theo các cột `by` (ví dụ `["gender"]` hoặc
`["dept", "gender"]`), bằng **tổng `admitted` / tổng `applied`** trong nhóm, không phải trung bình các tỉ
lệ. Tên Series: `admit_rate`.

`simpson_flip(df) -> bool`: `True` nếu khoảng cách giữa hai giới **đảo chiều** khi đi từ tỉ lệ gộp sang tỉ
lệ đã khử cơ cấu khoa. Tỉ lệ khử cơ cấu của một giới là trung bình có trọng số các tỉ lệ theo khoa của giới
đó, trọng số là **tổng số hồ sơ của khoa** (cả hai giới), giống nhau cho hai giới (direct
standardization). Đảo chiều = hai hiệu số `rate(giới 1) - rate(giới 2)` khác dấu và đều khác 0. `df` phải
có đúng hai giá trị `gender`, nếu không `raise ValueError`.

Vì sao không đếm "bao nhiêu khoa nữ cao hơn"? Vì khoa 25 hồ sơ và khoa 900 hồ sơ sẽ có cùng một phiếu.
Trọng số theo quy mô khoa trả lời câu hỏi: "nếu nam và nữ nộp vào các khoa theo **cùng** một cơ cấu, ai
được nhận nhiều hơn?"

In [ ]:
BERKELEY = pd.DataFrame(
    [  # khoa, giới, nộp, nhận (Bickel, Hammel & O'Connell 1975; R: UCBAdmissions)
        ("A", "men", 825, 512), ("A", "women", 108, 89),
        ("B", "men", 560, 353), ("B", "women", 25, 17),
        ("C", "men", 325, 120), ("C", "women", 593, 202),
        ("D", "men", 417, 138), ("D", "women", 375, 131),
        ("E", "men", 191, 53), ("E", "women", 393, 94),
        ("F", "men", 373, 22), ("F", "women", 341, 24),
    ],
    columns=["dept", "gender", "applied", "admitted"],
)


def admit_rate(df: pd.DataFrame, by: list[str]) -> pd.Series:
    """Tổng admitted / tổng applied theo các cột by, tên "admit_rate"."""
    # TODO
    raise NotImplementedError


def simpson_flip(df: pd.DataFrame) -> bool:
    """True nếu hiệu tỉ lệ gộp giữa hai giới và hiệu tỉ lệ khử cơ cấu khoa khác dấu."""
    # TODO: kiểm đúng 2 giới; tỉ lệ gộp; tỉ lệ theo khoa (unstack); trọng số = tổng applied mỗi khoa
    raise NotImplementedError

In [ ]:
# explore: tỉ lệ gộp, tỉ lệ theo khoa, và nữ nộp hồ sơ vào đâu
pooled = admit_rate(BERKELEY, ["gender"])
by_dept = admit_rate(BERKELEY, ["dept", "gender"]).unstack("gender")
applied = BERKELEY.pivot(index="dept", columns="gender", values="applied")
mix = applied / applied.sum()                                   # cơ cấu hồ sơ của mỗi giới theo khoa
dept_rate = admit_rate(BERKELEY, ["dept"])                      # độ "khó" của khoa, hai giới gộp

weights = applied.sum(axis=1)                                   # tổng hồ sơ mỗi khoa, chung cho hai giới
adjusted = by_dept.mul(weights, axis=0).sum() / weights.sum()
women_higher = int((by_dept.women > by_dept.men).sum())

print("tỉ lệ gộp:", pooled.round(3).to_dict())
print(pd.concat({"tỉ lệ nhận": by_dept, "cơ cấu hồ sơ": mix, "cả khoa": dept_rate.to_frame("tỉ lệ nhận")},
                axis=1).round(3).to_string())
print("số khoa nữ cao hơn nam:", women_higher, "/", len(by_dept))
print("tỉ lệ khử cơ cấu khoa:", adjusted.round(3).to_dict())
print("đảo chiều:", simpson_flip(BERKELEY))

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 3]})
colors = {"men": "#4c78a8", "women": "#f58518"}
a.bar(pooled.index, pooled.values * 100, color=[colors[g] for g in pooled.index])
a.set_ylim(0, 100)
a.set_ylabel("% hồ sơ được nhận")
a.set_xlabel("giới")
a.set_title("Gộp 6 khoa", loc="left")
x = np.arange(len(by_dept))
for i, g in enumerate(["men", "women"]):
    b.bar(x + (i - 0.5) * 0.4, by_dept[g] * 100, width=0.4, color=colors[g], label=g)
    for xi, n in zip(x, applied[g]):
        b.annotate(f"n={n}", (xi + (i - 0.5) * 0.4, 1), ha="center", fontsize=7, rotation=90, color="white")
b.set_xticks(x, by_dept.index)
b.set_ylim(0, 100)
b.set_xlabel("khoa (sắp theo tỉ lệ nhận chung giảm dần)")
b.set_ylabel("% hồ sơ được nhận")
b.legend(frameon=False)
b.set_title(f"Theo từng khoa: nữ cao hơn ở {women_higher}/{len(by_dept)} khoa", loc="left")
for ax in (a, b):
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

Đọc hai chart theo quy trình 5 bước. Bước 4 (so với tham chiếu) là chỗ mấu chốt: bar "gộp" so nam với
nữ, nhưng hai nhóm nộp vào **các khoa khác nhau**. Nhìn cột `cơ cấu hồ sơ`: phần lớn nam nộp vào khoa nào,
phần lớn nữ nộp vào khoa nào, và tỉ lệ nhận chung của các khoa đó ra sao? Đó là cơ chế của nghịch lý.

Lưu ý phạm vi: phân tầng theo khoa loại bỏ được nhiễu do **khoa**, chưa chứng minh không có thiên lệch ở
khâu khác (vì sao nữ nộp vào khoa khó, chất lượng hồ sơ). Bickel et al. kết luận cẩn trọng đúng như vậy.

## Phần B. Funnel đơn hàng Olist

`orders` (bảng `olist_orders_dataset.csv`) có 4 mốc thời gian cho mỗi đơn:

| Bước | Cột |
| --- | --- |
| `purchased` | `order_purchase_timestamp` |
| `approved` | `order_approved_at` |
| `carrier` | `order_delivered_carrier_date` |
| `delivered` | `order_delivered_customer_date` |

`order_funnel(orders) -> pd.Series`: số đơn có giá trị (không null) ở từng cột, index đúng 4 bước theo thứ
tự trên, tên Series `orders`. Đếm **từng cột độc lập**, đúng như dữ liệu ghi nhận (dữ liệu thật có vài đơn
thiếu mốc giữa nhưng có mốc sau; phần explore cho bạn thấy bao nhiêu).

`step_conversion(funnel) -> pd.Series`: tỉ lệ mỗi bước so với **bước trước**; bước đầu tiên là `1.0`. Cùng
index với `funnel`.

In [ ]:
FUNNEL_STEPS = {
    "purchased": "order_purchase_timestamp",
    "approved": "order_approved_at",
    "carrier": "order_delivered_carrier_date",
    "delivered": "order_delivered_customer_date",
}


def order_funnel(orders: pd.DataFrame) -> pd.Series:
    """Số đơn có mốc thời gian không null ở mỗi bước, tên "orders"."""
    # TODO
    raise NotImplementedError


def step_conversion(funnel: pd.Series) -> pd.Series:
    """Tỉ lệ so với bước trước; bước đầu = 1.0."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
path = ROOT / "data" / "raw" / "olist_orders_dataset.csv"
if not path.exists():
    raise FileNotFoundError("Chưa có Olist thật. Chạy: uv run python data/download_olist.py")
orders = pd.read_csv(path, parse_dates=list(FUNNEL_STEPS.values()))
print(len(orders), "đơn,", orders.order_purchase_timestamp.min(), "→", orders.order_purchase_timestamp.max())

funnel = order_funnel(orders)
table = pd.DataFrame({"orders": funnel, "so_voi_dau": funnel / funnel.iloc[0],
                      "so_voi_buoc_truoc": step_conversion(funnel), "mat_o_buoc_nay": -funnel.diff()})
print(table.round(4).to_string())

In [ ]:
# explore: vẽ funnel: bar ngang có thứ tự, ghi cả hai loại tỉ lệ
fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(funnel.index, funnel.values / 1000, color="#4c78a8")
ax.invert_yaxis()
ax.bar_label(bars, labels=[f"{n:,} | {p:.1%} so với đầu | {s:.1%} so với bước trước"
                           for n, p, s in zip(funnel, table.so_voi_dau, table.so_voi_buoc_truoc)],
             padding=3, fontsize=8)
ax.set_xlim(0, funnel.max() / 1000 * 2)
ax.set_xlabel("số đơn (nghìn)")
ax.set_ylabel("bước")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Funnel đơn hàng Olist 09/2016-10/2018", loc="left")
plt.show()

In [ ]:
# explore: funnel thật có "lồng nhau" hoàn hảo không? Đơn chưa giao tới khách đang ở trạng thái nào?
cols = list(FUNNEL_STEPS.values())
print("có delivered nhưng thiếu approved:", int((orders[cols[3]].notna() & orders[cols[1]].isna()).sum()))
print("có carrier nhưng thiếu approved:", int((orders[cols[2]].notna() & orders[cols[1]].isna()).sum()))
print("\nđơn chưa có mốc delivered, theo order_status:")
print(orders[orders[cols[3]].isna()].order_status.value_counts().to_string())

# kỳ dở dang: tỉ lệ đã giao theo tháng mua
by_month = (orders.assign(m=orders.order_purchase_timestamp.dt.to_period("M"))
            .groupby("m")[cols].apply(lambda d: d.notna().sum()))
by_month["da_giao"] = by_month[cols[3]] / by_month[cols[0]]
print(by_month[[cols[0], cols[3], "da_giao"]].tail(6).round(3).to_string())

## Đọc funnel, rồi làm trắc nghiệm

1. **Trục và phạm vi**: funnel gộp mọi đơn từ 09/2016 tới 10/2018. Những tháng cuối có đơn nào kịp giao
   chưa? Đơn mua tuần trước "chưa giao" có phải là đơn bị mất không?
2. **Bước mất nhiều nhất**: theo số tuyệt đối và theo tỉ lệ so với bước trước. Hai cách có cho cùng một
   câu trả lời không?
3. **Chất lượng dữ liệu**: đơn có mốc giao nhưng thiếu mốc duyệt là lỗi ghi nhận hay một luồng nghiệp vụ?
4. **Câu hỏi tiếp theo cho PM**: đơn dừng giữa funnel đang ở trạng thái nào (`order_status`)? Phần nào là
   huỷ thật, phần nào đang trên đường?

Viết headline cho funnel mà PM vận hành sẽ đọc trong 5 giây.

Xong thì mở app, làm phần **trắc nghiệm** của bài.